### 1.1 Define the Business Process and the Fact Grain

**Business Process:** The main process we are analyzing is the daily activity of an investment account, specifically looking at the financial transactions made during the year 2024.

**Fact Grain:** The grain is the level of detail of our central table. As requested in the instructions, we are keeping the lowest level of detail: one row in the `Fact_Transactions` table represents exactly one single transaction (like a BUY or a SELL) from the original account statement file.

### 1.2 Identify Fact and Dimensions

To design a clean star schema, attributes were selected based on their relevance for the analysis. Redundant or empty columns from the source files were intentionally excluded to keep the dimension tables optimized.

* **Fact Table: `Fact_Transactions`**
    * **Attributes:** `IDTransaction`, `Unit` (Quantity)
    * **Foreign Keys:** `time_key`, `geo_key`, `symbol_key`, `type_key`
    * *(Note: The empty 'Unnamed: 5' column from the raw file was discarded).*

* **Dimension Table: `Dim_Time`**
    * **Attributes:** `time_key` (Surrogate Key), `Date` (Full datetime format), `Day`, `Month`, `Quarter`, `Year`

* **Dimension Table: `Dim_Geography`**
    * **Attributes:** `geo_key` (Surrogate Key), `name` (Country), `alpha-3` (Standard country code), `region`, `sub-region`
    * *(Note: Redundant attributes like 'alpha-2', 'country-code', and 'region-code' were excluded as 'alpha-3' and the textual names are sufficient for geographical analysis).*

* **Dimension Table: `Dim_Symbol`**
    * **Attributes:** `symbol_key` (Surrogate Key), `symbol` (Ticker), `company_name`, `sector`, `industry`

* **Dimension Table: `Dim_TransactionType`**
    * **Attributes:** `type_key` (Surrogate Key), `TransactionType` (BUY, SELL)

### 1.3 Define Dimension Hierarchies

Hierarchies are useful to group data together, moving from small details to bigger categories. Based on our files and the columns we kept, here are the logical hierarchies for our dimensions:

* **Dim_Time:** `Day -> Month -> Quarter -> Year` (The standard way to group dates).
* **Dim_Geography:** `Country -> Region -> Sub-region` (As indicated in the instructions).
* **Dim_Symbol:** `Symbol (Company) -> Industry -> Sector` (Going from the specific company to its general market sector).
* **Dim_TransactionType:** This dimension is flat. There is no hierarchy here because a transaction type (like BUY or SELL) cannot be grouped into a larger category.

### 1.4 Design the Star Schema

The final star schema is designed with one central Fact table (`fact_transactions`) and four Dimension tables (`dim_geography`, `dim_symbol`, `dim_transaction`, and `dim_time`).

* **Surrogate Keys:** Artificial integer keys (`geo_key`, `symbol_key`, `type_key`, `time_key`) were created for each dimension to uniquely identify records and optimize database performance.
* **Foreign Keys Mapping:** The Fact table was joined to the Dimensions by matching the original text values (like Date or Symbol). We also used a temporary step to find the Country from the Symbol. Finally, we deleted all the text columns to keep only the numerical keys (geo_key, time_key...).
* **Measures and Attributes:** The quantitative measures kept in the Fact table are `Unit` and `IDTransaction`. All descriptive attributes (such as country name, region, company sector...) were successfully removed from the Fact table and placed into their respective dimension tables to avoid unnecessary duplication.

In [1]:
# =====================================================================
# 2.1 LOAD AND CLEAN THE DATA
# =====================================================================

import pandas as pd

# Read the data files into dataframes
fact_raw = pd.read_csv('account-statement-1-1-2024-12-31-2024.csv', sep=';')
fact_raw = fact_raw.dropna(subset=['IDTransaction']).reset_index(drop=True)
dim_sym_raw = pd.read_csv('symbols.csv', sep=';')
dim_geo_raw = pd.read_csv('country.csv')

# Fix the lazy country names in the symbols file to match the official ISO file
country_corrections = {
    'United States': 'United States of America',
    'USA': 'United States of America',
    'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland',
    'UK': 'United Kingdom of Great Britain and Northern Ireland',
    'Russia': 'Russian Federation',
    'South Korea': 'Korea, Republic of',
    'Taiwan': 'Taiwan, Province of China'
}
dim_sym_raw['country'] = dim_sym_raw['country'].replace(country_corrections)

# Build dim_geography (with duplicate safety)
dim_geography = dim_geo_raw[['name', 'alpha-3', 'region', 'sub-region']].drop_duplicates(subset=['name']).reset_index(drop=True)
dim_geography = dim_geography.rename(columns={'name': 'country'})
dim_geography['geo_key'] = dim_geography.index + 1

# Build dim_symbol (with duplicate safety)
dim_symbol = dim_sym_raw[['symbol', 'company_name', 'sector', 'industry']].drop_duplicates(subset=['symbol']).reset_index(drop=True)
dim_symbol['symbol_key'] = dim_symbol.index + 1

# Build dim_transaction
dim_transaction = fact_raw[['TransactionType']].drop_duplicates().reset_index(drop=True)
dim_transaction['type_key'] = dim_transaction.index + 1

# Build dim_time
fact_raw['Date'] = pd.to_datetime(fact_raw['Date'], dayfirst=True, format='mixed')
dim_time = pd.DataFrame({'full_date': fact_raw['Date'].unique()})
dim_time['day'] = dim_time['full_date'].dt.day
dim_time['month'] = dim_time['full_date'].dt.month
dim_time['quarter'] = dim_time['full_date'].dt.quarter
dim_time['year'] = dim_time['full_date'].dt.year
dim_time['time_key'] = dim_time.index + 1

fact_transactions = fact_raw.copy()

# Map keys to the fact dataframe
fact_transactions = fact_transactions.merge(dim_time[['full_date', 'time_key']], left_on='Date', right_on='full_date', how='left')
fact_transactions = fact_transactions.merge(dim_transaction[['TransactionType', 'type_key']], on='TransactionType', how='left')
fact_transactions = fact_transactions.merge(dim_symbol[['symbol', 'symbol_key']], left_on='Symbol', right_on='symbol', how='left')

# Get country from symbol to map geo key
symbol_to_country = dim_sym_raw[['symbol', 'country']].drop_duplicates(subset=['symbol'])
fact_transactions = fact_transactions.merge(symbol_to_country, left_on='Symbol', right_on='symbol', how='left')
fact_transactions = fact_transactions.merge(dim_geography[['country', 'geo_key']], on='country', how='left')

# Drop textual columns
columns_to_drop = ['Date', 'TransactionType', 'Symbol', 'Unnamed: 5', 'full_date', 'country', 'symbol', 'symbol_x', 'symbol_y']
fact_transactions = fact_transactions.drop(columns=columns_to_drop, errors='ignore')

# Data quality checks
print("--- Data quality checks ---")
missing_symbols = fact_transactions['symbol_key'].isna().sum()
missing_countries = fact_transactions['geo_key'].isna().sum()

# Verify that every transaction symbol exists in the symbols dataset
print(f"Orphan Symbols found: {missing_symbols}")

# Verify that every company country can be mapped to the country dataset
print(f"Orphan Countries found: {missing_countries}")

if missing_symbols > 0 or missing_countries > 0:
    print("\nAction taken: Dropping transactions with unmapped symbols/countries to maintain Star Schema integrity.")
    fact_transactions = fact_transactions.dropna(subset=['symbol_key', 'geo_key']).reset_index(drop=True)

print("\n--- Final Quality Report ---")
print(f"Missing Symbols in Fact Table: {fact_transactions['symbol_key'].isna().sum()} (Expected: 0)")
print(f"Missing Countries in Fact Table: {fact_transactions['geo_key'].isna().sum()} (Expected: 0)")

print(f"Total Transactions processed successfully: {len(fact_transactions)}")

# Display columns and preview
print("Final Fact Table Columns:", fact_transactions.columns.tolist())
display(fact_transactions.head())

--- Data quality checks ---
Orphan Symbols found: 212
Orphan Countries found: 212

Action taken: Dropping transactions with unmapped symbols/countries to maintain Star Schema integrity.

--- Final Quality Report ---
Missing Symbols in Fact Table: 0 (Expected: 0)
Missing Countries in Fact Table: 0 (Expected: 0)
Total Transactions processed successfully: 2069
Final Fact Table Columns: ['IDTransaction', 'Unit', 'time_key', 'type_key', 'symbol_key', 'geo_key']


,IDTransaction,Unit,time_key,type_key,symbol_key,geo_key
0,2.769834e+09,1605.0,1,1,284.0,175.0
1,2.767325e+09,1605.0,2,2,284.0,175.0
2,2.815474e+09,914.0,3,2,284.0,175.0
3,2.622244e+09,646.0,4,1,4.0,25.0
4,2.629871e+09,646.0,5,2,258.0,131.0


In [2]:
# =====================================================================
# 2.2 Analytical Questions
# =====================================================================

import sqlite3

# Setting up a temporary in-memory SQL database for my analysis
conn = sqlite3.connect(':memory:')

# Loading my cleaned and mapped DataFrames into the SQL engine
fact_transactions.to_sql('fact_transactions', conn, index=False)
dim_symbol.to_sql('dim_symbol', conn, index=False)
dim_transaction.to_sql('dim_transaction', conn, index=False)
dim_geography.to_sql('dim_geography', conn, index=False)
dim_time.to_sql('dim_time', conn, index=False)

# Defining my 5 selected sql queries
# Query 1: Filtering on USA, SELL type, and 2024, then grouping by sector.
q1 = """
SELECT s.sector, COUNT(f.IDTransaction) AS number_of_sells
FROM fact_transactions AS f
INNER JOIN dim_symbol AS s ON f.symbol_key = s.symbol_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'SELL' AND g."alpha-3" = 'USA' AND t.year = 2024
GROUP BY s.sector
ORDER BY number_of_sells DESC
LIMIT 5;
"""

# Query 2: Joining with time dimension only, grouping by quarter.
q2 = """
SELECT t.quarter, COUNT(f.IDTransaction) AS total_transactions
FROM fact_transactions AS f
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE t.year = 2024
GROUP BY t.quarter
ORDER BY total_transactions DESC;
"""

# Query 3: Filtering on SELL and 2024, then grouping by country name.
q3 = """
SELECT g.country, COUNT(f.IDTransaction) AS number_of_sells
FROM fact_transactions AS f
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'SELL' AND t.year = 2024
GROUP BY g.country
ORDER BY number_of_sells DESC
LIMIT 10;
"""

# Query 4: Using SUM() on the 'Unit' measure instead of COUNT(), filtering on BUY.
q4 = """
SELECT g.region, SUM(f.Unit) AS total_units_bought
FROM fact_transactions AS f
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'BUY' AND t.year = 2024
GROUP BY g.region
ORDER BY total_units_bought DESC
LIMIT 5;
"""

# Query 5: Joining Fact table with symbol and time, grouping by company symbol.
q5 = """
SELECT s.symbol, COUNT(f.IDTransaction) AS total_transactions
FROM fact_transactions AS f
INNER JOIN dim_symbol AS s ON f.symbol_key = s.symbol_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE t.year = 2024
GROUP BY s.symbol
ORDER BY total_transactions DESC
LIMIT 10;
"""

# Executing queries and displaying my results
print("Q1: Top 5 sectors by SELL transactions in USA (2024)")
display(pd.read_sql_query(q1, conn))

print("\nQ2: Quarters of 2024 ranked by total transactions")
display(pd.read_sql_query(q2, conn))

print("\nQ3: Top 10 countries by SELL transactions (2024)")
display(pd.read_sql_query(q3, conn))

print("\nQ4: Top 5 regions by total units bought (2024)")
display(pd.read_sql_query(q4, conn))

print("\nQ5: Top 10 symbols by total transactions (2024)")
display(pd.read_sql_query(q5, conn))

Q1: Top 5 sectors by SELL transactions in USA (2024)


,sector,number_of_sells
0,Technology,158
1,Communication Services,58
2,Financial Services,55
3,Healthcare,50
4,Consumer Cyclical,48



Q2: Quarters of 2024 ranked by total transactions


,quarter,total_transactions
0,1,999
1,2,542
2,3,268
3,4,260



Q3: Top 10 countries by SELL transactions (2024)


,country,number_of_sells
0,United States of America,389
1,United Kingdom of Great Britain and Northern I...,130
2,China,112
3,Brazil,69
4,"Taiwan, Province of China",50
5,"Netherlands, Kingdom of the",46
6,Switzerland,37
7,Ireland,31
8,Luxembourg,27
9,Canada,22



Q4: Top 5 regions by total units bought (2024)


,region,total_units_bought
0,Americas,37026.0
1,Europe,22528.0
2,Asia,9198.0
3,None,2339.0



Q5: Top 10 symbols by total transactions (2024)


,symbol,total_transactions
0,ARM,100
1,AMD,97
2,TSM,80
3,TIMB,76
4,GOOG,52
5,MSFT,49
6,AMZN,47
7,ARDX,43
8,BRFS,42
9,BLK,42


### 1. What are the top 5 sectors by number of SELL transactions in US during 2024?

**Explanation of the query logic:**
I used `INNER JOIN` to link the Fact table with the Time, Geography, Symbol, and Transaction dimensions. I applied `WHERE` clauses to filter for the year 2024, the 'USA' country code, and 'SELL' transactions. Finally, I grouped the records by sector using `GROUP BY`, counted the number of transactions with `COUNT()`, and sorted the result in descending order using `ORDER BY ... DESC` with a `LIMIT` of 5.

**SQL Query:**
```sql
SELECT s.sector, COUNT(f.IDTransaction) AS number_of_sells
FROM fact_transactions AS f
INNER JOIN dim_symbol AS s ON f.symbol_key = s.symbol_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'SELL' AND g."alpha-3" = 'USA' AND t.year = 2024
GROUP BY s.sector
ORDER BY number_of_sells DESC
LIMIT 5;

### 2. Rank all quarters of 2024 by total number of transactions (BUY + SELL).

**Explanation of the query logic:**
To find this ranking, I joined the Fact table exclusively with the time dimension. I filtered the data for the year 2024. Then, I grouped the results by `quarter` and used the `COUNT()` function to calculate the total number of transactions (which includes both BUY and SELL). The results are ranked highest to lowest using `ORDER BY`.

**SQL Query:**
```sql
SELECT t.quarter, COUNT(f.IDTransaction) AS total_transactions
FROM fact_transactions AS f
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE t.year = 2024
GROUP BY t.quarter
ORDER BY total_transactions DESC;

### 3. What are the top 10 countries by number of SELL transactions in 2024?

**Explanation of the query logic:**
I joined the Fact table with the geography, time, and transaction dimensions. I applied a `WHERE` filter to isolate 'SELL' transactions happening in 2024. I grouped the output by `country` name, counted the transactions, and ordered them in descending order, limiting the output to the top 10.

**SQL Query:**
```sql
SELECT g.country, COUNT(f.IDTransaction) AS number_of_sells
FROM fact_transactions AS f
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'SELL' AND t.year = 2024
GROUP BY g.country
ORDER BY number_of_sells DESC
LIMIT 10;

### 4. What are the top 5 regions by total units bought in 2024?

**Explanation of the query logic:**
Since the question asks for "units bought" and not "number of transactions", I joined the Fact table with geography, time, and transaction dimensions. I filtered for 'BUY' events in 2024. I grouped the data by geographic `region`, and crucially used the `SUM(f.Unit)` aggregate function to calculate the total volume, ordering the result to keep only the top 5.

**SQL Query:**
```sql
SELECT g.region, SUM(f.Unit) AS total_units_bought
FROM fact_transactions AS f
INNER JOIN dim_geography AS g ON f.geo_key = g.geo_key
INNER JOIN dim_transaction AS tr ON f.type_key = tr.type_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE tr.TransactionType = 'BUY' AND t.year = 2024
GROUP BY g.region
ORDER BY total_units_bought DESC
LIMIT 5;

### 5. What are the top 10 symbols by number of transactions (BUY + SELL) in 2024?

**Explanation of the query logic:**
To rank the most active symbols, I joined the Fact table with the symbol and time dimensions. After filtering for the year 2024, I grouped the dataset by `symbol`. I used the `COUNT()` function to measure the frequency of transactions for each company, ordering the results from highest to lowest and limiting to the top 10.

**SQL Query:**
```sql
SELECT s.symbol, COUNT(f.IDTransaction) AS total_transactions
FROM fact_transactions AS f
INNER JOIN dim_symbol AS s ON f.symbol_key = s.symbol_key
INNER JOIN dim_time AS t ON f.time_key = t.time_key
WHERE t.year = 2024
GROUP BY s.symbol
ORDER BY total_transactions DESC
LIMIT 10;